In [1]:
%pip install -q "torch==2.3.1" "transformers==4.40.2" pysentiment2 pandas pyarrow tqdm

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
try:
    import torch

    if torch.backends.mps.is_available():
        DEVICE = "mps"
    elif torch.cuda.is_available():
        DEVICE = "cuda"
    else:
        DEVICE = "cpu"

    print(f"Using device: {DEVICE}")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Using device: mps


In [3]:
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    TOKENIZER = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    MODEL = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(DEVICE)
    MODEL.eval()
    print("FinBERT loaded")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

/Users/dhruvansh/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/dhruvansh/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/dhruvansh/Library/Python/3.9/lib/python/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/dhruvansh/Library/Python/3.9/lib/python/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will

FinBERT loaded


In [4]:
try:
    import pysentiment2 as ps

    _lm = ps.LM()
    LM_POS = _lm._posset
    LM_NEG = _lm._negset
    print(f"LM word lists loaded: {len(LM_POS)} positive, {len(LM_NEG)} negative words")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

LM word lists loaded: 140 positive, 893 negative words


In [5]:
try:
    import pandas as pd
    from pathlib import Path

    if not Path("data/raw_news.parquet").exists():
        raise FileNotFoundError("data/raw_news.parquet not found — run notebook_01 first")

    df_news = pd.read_parquet("data/raw_news.parquet")

    null_mask = df_news["headline"].isnull() | (df_news["headline"].str.strip() == "")
    n_dropped = null_mask.sum()
    if n_dropped > 0:
        df_news = df_news[~null_mask].copy()
        print(f"Dropped {n_dropped} rows with null/empty headlines")

    print(f"Loaded {len(df_news)} headlines across {df_news['ticker'].nunique()} tickers")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Loaded 73936 headlines across 10 tickers


In [6]:
try:
    def lm_score(text: str) -> float:
        if not isinstance(text, str) or not text.strip():
            return 0.0
        tokens = text.lower().split()
        if not tokens:  # explicit zero-token guard — prevents ZeroDivisionError
            return 0.0
        pos = sum(1 for t in tokens if t in LM_POS)
        neg = sum(1 for t in tokens if t in LM_NEG)
        return max(-1.0, min(1.0, (pos - neg) / len(tokens)))

    print("lm_score defined")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

lm_score defined


In [7]:
try:
    import torch
    import torch.nn.functional as F

    def finbert_scores(texts: list) -> list:
        def _run_on_device(texts, device):
            inputs = TOKENIZER(
                texts,
                max_length=512,
                truncation=True,
                padding=True,
                return_tensors="pt",
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                logits = MODEL(**inputs).logits
            probs = F.softmax(logits.cpu(), dim=1).numpy()
            scores = []
            for row in probs:
                idx = int(row.argmax())
                if idx == 0:
                    scores.append(float(row[0]))
                elif idx == 1:
                    scores.append(-float(row[1]))
                elif idx == 2:
                    scores.append(0.0)
                else:
                    print(f"WARNING: unknown FinBERT label {idx}")
                    scores.append(0.0)
            return scores

        try:
            return _run_on_device(texts, DEVICE)
        except NotImplementedError:
            print(f"WARNING: NotImplementedError on {DEVICE}, retrying on CPU")
            return _run_on_device(texts, "cpu")

    print("finbert_scores defined")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

finbert_scores defined


In [8]:
try:
    import numpy as np
    from tqdm import tqdm

    BATCH_SIZE = 64
    all_finbert = []
    all_lm = []

    headlines = df_news["headline"].fillna("").tolist()

    for start in tqdm(range(0, len(df_news), BATCH_SIZE), desc="Scoring batches"):
        batch_texts = headlines[start : start + BATCH_SIZE]
        all_finbert.extend(finbert_scores(batch_texts))
        all_lm.extend(lm_score(t) for t in batch_texts)

    df_news["finbert_score"] = all_finbert
    df_news["lm_score"] = all_lm
    print(f"Scored {len(df_news)} headlines")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Scoring batches: 100%|██████████| 1156/1156 [03:41<00:00,  5.23it/s]

Scored 73936 headlines


In [9]:
try:
    df_news["final_score"] = 0.70 * df_news["finbert_score"] + 0.30 * df_news["lm_score"]

    print("Sample spot-check (3 rows):")
    print(df_news[["headline", "finbert_score", "lm_score", "final_score"]].sample(3).to_string())
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Sample spot-check (3 rows):
                                                               headline  finbert_score  lm_score  final_score
47941  An old church was given a second life as a stunning events space       0.000000       0.0     0.000000
21838          Man arrested for burglarizing Patriots Gronkowski's home      -0.772522       0.0    -0.540765
71776              Apple says reopening all its branded stores in China       0.000000       0.0     0.000000


In [10]:
try:
    df_daily = (
        df_news.groupby(["date", "ticker"])
        .agg(
            sentiment_score=("final_score", "mean"),
            article_count=("final_score", "count"),
            avg_finbert_score=("finbert_score", "mean"),
            avg_lm_score=("lm_score", "mean"),
        )
        .reset_index()
    )

    df_daily = df_daily[["date", "ticker", "sentiment_score", "article_count", "avg_finbert_score", "avg_lm_score"]]
    df_daily["article_count"] = df_daily["article_count"].astype("int64")
    print(df_daily.shape)
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

(7817, 6)


In [11]:
try:
    import os

    os.makedirs("data", exist_ok=True)
    df_daily.to_parquet("data/sentiment_daily.parquet", index=False)
    print(f"Saved data/sentiment_daily.parquet — {len(df_daily)} stock-day rows")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Saved data/sentiment_daily.parquet — 7817 stock-day rows


In [12]:
try:
    df_out = pd.read_parquet("data/sentiment_daily.parquet")
    print(f"Total headlines processed: {len(df_news)}")
    print(f"Stock-day rows in output: {len(df_out)}")
    print(f"Date range: {df_out['date'].min().date()} to {df_out['date'].max().date()}")
except Exception as e:
    print(f"\n[ERROR] {type(e).__name__}: {e}")
    raise SystemExit(1) from None

Total headlines processed: 73936
Stock-day rows in output: 7817
Date range: 2016-01-01 to 2020-04-01
